# 1. Minimal manual tutorial
In this notebook, we will walk through a minimal example of how to use the ASSUME framework. We will first initialize the world instance, next we will create a single market and its operator, afterwards we will add a generation and a demand agent, and finally start the simulation.

## Setting Up the Simulation Environment

Here we just install the ASSUME core package via pip. The instructions for an installation can be found here: https://assume.readthedocs.io/en/latest/installation.html.

This step is only required if you are working with this notebook in collab. If you are working locally and you have installed the assume package, you can skip this step.

In [1]:
import importlib.util

# Check whether notebook is run in google colab
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    !pip install assume-framework

First, let's set up the necessary environment and import the required libraries.

In [4]:
import logging
import os
from datetime import datetime, timedelta

import pandas as pd
from dateutil import rrule as rr

from assume import World
from assume.common.forecaster import UnitForecaster
from assume.common.market_objects import MarketConfig, MarketProduct

log = logging.getLogger(__name__)

os.makedirs("local_db", exist_ok=True)

db_uri = "sqlite:///local_db/assume_db.db"

world = World(database_uri=db_uri)

start = datetime(2023, 10, 4)
end = datetime(2023, 12, 5)
index = pd.date_range(
    start=start,
    end=end + timedelta(hours=24),
    freq="h",
)
simulation_id = "world_script_simulation"

INFO:assume.world:Connected to the database


In [5]:
print(type(world))

<class 'assume.world.World'>


In this section, we begin by importing the necessary libraries and modules. Additionally, we define the database URI. For this instance, we will utilize a local SQLite database to store our results. In subsequent notebooks, we will transition to using a timescaledb database to store the results, which can then be visualized using the included Grafana dashboards. 

Subsequently, we instantiate the `World` class, the primary class responsible for managing the simulation. We also establish the simulation's start and end dates, define the simulation index and step size, and assign a simulation ID. This unique identifier is crucial for referencing the simulation in the database.

## Initializing the Simulation
Next, we initialize the simulation by executing the setup function. The setup function sets up the environment for the simulation. It initializes various parameters and components required for the simulation run, including the clock, learning configuration, forecaster, container, connection type, and output agents.

This is a function to store and compare instances of the world object in order to see the changes as attributes of the market are defined.

In [6]:
# snapshot attributes before/after
def snapshot_world(world, keys=None):
    if keys is None:
        keys = ["start","end","simulation_id","index","markets","unit_operators","container","clock","database_uri","db_uri","engine"]
    snap = {}
    for k in keys:
        val = getattr(world, k, None)
        # small helper to avoid huge prints
        if isinstance(val, (dict, list, set)):
            snap[k] = {"type": type(val).__name__, "len": len(val)}
        else:
            try:
                snap[k] = {"type": type(val).__name__, "repr": repr(val)}
            except Exception:
                snap[k] = {"type": type(val).__name__, "repr": "<unreprable>"}
    return snap

before = snapshot_world(world)
from pprint import pprint
print("BEFORE")
pprint(before, width=120)

BEFORE
{'clock': {'repr': 'None', 'type': 'NoneType'},
 'container': {'repr': 'None', 'type': 'NoneType'},
 'database_uri': {'repr': 'None', 'type': 'NoneType'},
 'db_uri': {'repr': 'sqlite:///local_db/assume_db.db', 'type': 'URL'},
 'end': {'repr': 'None', 'type': 'NoneType'},
 'engine': {'repr': 'None', 'type': 'NoneType'},
 'index': {'repr': 'None', 'type': 'NoneType'},
 'markets': {'len': 0, 'type': 'dict'},
 'simulation_id': {'repr': 'None', 'type': 'NoneType'},
 'start': {'repr': 'None', 'type': 'NoneType'},
 'unit_operators': {'len': 0, 'type': 'dict'}}


In [7]:
world.setup(
    start=start,
    end=end,
    save_frequency_hours=48,
    simulation_id=simulation_id,
)

In [8]:
# run your setup or changes, then:
after = snapshot_world(world)
print("\nAFTER")
pprint(after, width=120)

# quick diff of changed keys
changed = [k for k in after.keys() if before.get(k)!=after.get(k)]
print("\nChanged keys:", changed)
for k in changed:
    print(f"\n--- {k} ---\nBEFORE: {before[k]}\nAFTER : {after[k]}")


AFTER
{'clock': {'repr': '<mango.util.clock.ExternalClock object at 0x14cce2540>', 'type': 'ExternalClock'},
 'container': {'repr': '<mango.container.external_coupling.ExternalSchedulingContainer object at 0x14cce0380>',
               'type': 'ExternalSchedulingContainer'},
 'database_uri': {'repr': 'None', 'type': 'NoneType'},
 'db_uri': {'repr': 'sqlite:///local_db/assume_db.db', 'type': 'URL'},
 'end': {'repr': 'datetime.datetime(2023, 12, 5, 0, 0)', 'type': 'datetime'},
 'engine': {'repr': 'None', 'type': 'NoneType'},
 'index': {'repr': 'None', 'type': 'NoneType'},
 'markets': {'len': 0, 'type': 'dict'},
 'simulation_id': {'repr': "'world_script_simulation'", 'type': 'str'},
 'start': {'repr': 'datetime.datetime(2023, 10, 4, 0, 0)', 'type': 'datetime'},
 'unit_operators': {'len': 0, 'type': 'dict'}}

Changed keys: ['start', 'end', 'simulation_id', 'container', 'clock']

--- start ---
BEFORE: {'type': 'NoneType', 'repr': 'None'}
AFTER : {'type': 'datetime', 'repr': 'datetime.datet

## Configuring market
Here, we define a market configuration, set up a market operator, and add the configured market to the simulation world.

In [7]:
from dateutil.relativedelta import relativedelta as rd

marketdesign = [
    MarketConfig(
        market_id="EOM",
        #opening_hours=rr.rrule(rr.HOURLY, interval=24, dtstart=start, until=end),
        opening_hours=rr.rrule(rr.DAILY, byhour=12, dtstart=start, until=end),
        opening_duration=timedelta(hours=1),
        market_mechanism="pay_as_clear",
        market_products=[MarketProduct(duration=timedelta(hours=1), count=24, first_delivery=timedelta(hours=12))],
        additional_fields=["block_id", "link", "exclusive_id"],
    )
]

This code segment sets up a market configuration named "EOM" with specific opening hours, market mechanism, products, and additional fields, providing the foundation for simulating and analyzing the behavior of this particular electricity market.

In this code:
- `marketdesign` is a list containing a single market configuration.

- `MarketConfig(...)` defines the configuration for a specific market. In this case, it's named "EOM" (Energy Only Market).

  - `name="EOM"` - Specifies the name of the market configuration as "EOM".

  - `opening_hours=rr.rrule(rr.HOURLY, interval=24, dtstart=start, until=end)` - Defines the opening hours for the market using a rule that repeats hourly with a 24-hour interval, starting at `start` and ending at `end`. This indicates that the market operates on a daily basis.

  - `opening_duration=timedelta(hours=1)` - Specifies the duration of each market opening as 1 hour.

  - `market_mechanism="pay_as_clear"` - Indicates the market mechanism used, in this case, "pay as clear", which is a common mechanism in electricity markets where all accepted bids are paid the market-clearing price.

  - `market_products=[MarketProduct(timedelta(hours=1), 24, timedelta(hours=1))]` - Defines the market products available. In this case, it seems to be a single product with a duration of 1 hour, 24 periods, and a period duration of 1 hour.

  - `additional_fields=["block_id", "link", "exclusive_id"]` - Specifies additional fields associated with this market configuration, such as "block_id", "link", and "exclusive_id".

In [8]:
# snapshot attributes before/after
before = snapshot_world(world)
from pprint import pprint
print("BEFORE")
pprint(before, width=120)

BEFORE
{'clock': {'repr': '<mango.util.clock.ExternalClock object at 0x148dc38f0>', 'type': 'ExternalClock'},
 'container': {'repr': '<mango.container.external_coupling.ExternalSchedulingContainer object at 0x148702c00>',
               'type': 'ExternalSchedulingContainer'},
 'database_uri': {'repr': 'None', 'type': 'NoneType'},
 'db_uri': {'repr': 'sqlite:///local_db/assume_db.db', 'type': 'URL'},
 'end': {'repr': 'datetime.datetime(2023, 12, 5, 0, 0)', 'type': 'datetime'},
 'engine': {'repr': 'None', 'type': 'NoneType'},
 'index': {'repr': 'None', 'type': 'NoneType'},
 'markets': {'len': 0, 'type': 'dict'},
 'simulation_id': {'repr': "'world_script_simulation'", 'type': 'str'},
 'start': {'repr': 'datetime.datetime(2023, 10, 4, 0, 0)', 'type': 'datetime'},
 'unit_operators': {'len': 0, 'type': 'dict'}}


In [9]:
mo_id = "market_operator"
world.add_market_operator(id=mo_id)

for market_config in marketdesign:
    world.add_market(market_operator_id=mo_id, market_config=market_config)

In [10]:
# run your setup or changes, then:
after = snapshot_world(world)
print("\nAFTER")
pprint(after, width=120)

# quick diff of changed keys
changed = [k for k in after.keys() if before.get(k)!=after.get(k)]
print("\nChanged keys:", changed)
for k in changed:
    print(f"\n--- {k} ---\nBEFORE: {before[k]}\nAFTER : {after[k]}")


AFTER
{'clock': {'repr': '<mango.util.clock.ExternalClock object at 0x148dc38f0>', 'type': 'ExternalClock'},
 'container': {'repr': '<mango.container.external_coupling.ExternalSchedulingContainer object at 0x148702c00>',
               'type': 'ExternalSchedulingContainer'},
 'database_uri': {'repr': 'None', 'type': 'NoneType'},
 'db_uri': {'repr': 'sqlite:///local_db/assume_db.db', 'type': 'URL'},
 'end': {'repr': 'datetime.datetime(2023, 12, 5, 0, 0)', 'type': 'datetime'},
 'engine': {'repr': 'None', 'type': 'NoneType'},
 'index': {'repr': 'None', 'type': 'NoneType'},
 'markets': {'len': 1, 'type': 'dict'},
 'simulation_id': {'repr': "'world_script_simulation'", 'type': 'str'},
 'start': {'repr': 'datetime.datetime(2023, 10, 4, 0, 0)', 'type': 'datetime'},
 'unit_operators': {'len': 0, 'type': 'dict'}}

Changed keys: ['markets']

--- markets ---
BEFORE: {'type': 'dict', 'len': 0}
AFTER : {'type': 'dict', 'len': 1}


In [11]:
# python
from pprint import pprint
for mid, m in getattr(world, "markets", {}).items():
    print("=== market key:", mid)
    pprint(getattr(m, "__dict__", {}), width=120)

=== market key: EOM
{'additional_fields': ['block_id', 'link', 'exclusive_id'],
 'addr': AgentAddress(protocol_addr='world', aid='market_operator'),
 'eligible_obligations_lambda': <function MarketConfig.<lambda> at 0x128b27b00>,
 'market_id': 'EOM',
 'market_mechanism': 'pay_as_clear',
 'market_products': [MarketProduct(duration=datetime.timedelta(seconds=3600),
                                   count=24,
                                   first_delivery=datetime.timedelta(seconds=43200),
                                   only_hours=None,
                                   eligible_lambda_function=None)],
 'maximum_bid_price': 3000.0,
 'maximum_bid_volume': 2000.0,
 'maximum_gradient': None,
 'minimum_bid_price': -500.0,
 'opening_duration': datetime.timedelta(seconds=3600),
 'opening_hours': <dateutil.rrule.rrule object at 0x148dc2990>,
 'param_dict': {},
 'price_tick': None,
 'price_unit': '€/MWh',
 'product_type': 'energy',
 'supports_get_unmatched': False,
 'volume_tick': None,


In [12]:
# python
from itertools import islice
from pprint import pprint

for mid, m in getattr(world, "markets", {}).items():
    print("\n=== market:", mid)
    # rrule may be stored on the market or on its config
    oh = getattr(m, "opening_hours", None) or getattr(getattr(m, "config", None), "opening_hours", None)
    if oh is None:
        print(" no opening_hours found")
        continue

    # human-readable first few openings
    print(" first openings (up to 10):")
    for dt in islice(oh, 10):
        print("  ", dt.isoformat())

    # full list (careful if very large)
    # all_opens = list(oh)
    # print([d.isoformat() for d in all_opens])

    # brief summary of rrule internals (best-effort, some are private)
    info = {
        "dtstart": getattr(oh, "_dtstart", None) or getattr(oh, "dtstart", None),
        "until": getattr(oh, "_until", None) or getattr(oh, "until", None),
        "freq": getattr(oh, "_freq", None),
        "interval": getattr(oh, "_interval", None),
        "byhour": getattr(oh, "_byhour", None),
        "byweekday": getattr(oh, "_byweekday", None),
    }
    pprint(info, width=120)

    # optional: output ISO strings to JSON-serializable list
    opens_iso = [d.isoformat() for d in islice(oh, 1000)]
    print(" sample_iso_count:", len(opens_iso))


=== market: EOM
 first openings (up to 10):
   2023-10-04T12:00:00
   2023-10-05T12:00:00
   2023-10-06T12:00:00
   2023-10-07T12:00:00
   2023-10-08T12:00:00
   2023-10-09T12:00:00
   2023-10-10T12:00:00
   2023-10-11T12:00:00
   2023-10-12T12:00:00
   2023-10-13T12:00:00
{'byhour': (12,),
 'byweekday': None,
 'dtstart': datetime.datetime(2023, 10, 4, 0, 0),
 'freq': 3,
 'interval': 1,
 'until': datetime.datetime(2023, 12, 5, 0, 0)}
 sample_iso_count: 62


In [13]:
from itertools import islice
from assume.common.utils import get_available_products

# Get all openings (assuming not too many for this example)
openings = list(islice(oh, 1000))
if openings:
    selected_openings = [openings[0]]  # first
    if len(openings) > 1:
        selected_openings.append(openings[-1])  # last

    for open_dt in selected_openings:
        print(" auction opens:", open_dt.isoformat())
        products = get_available_products(m.market_products, open_dt)
        for prod_idx, (flow_start, flow_end, only_hours) in enumerate(products):
            print(f"  product #{prod_idx}: {flow_start.isoformat()} -> {flow_end.isoformat()}")
        print()

 auction opens: 2023-10-04T12:00:00
  product #0: 2023-10-05T00:00:00 -> 2023-10-05T01:00:00
  product #1: 2023-10-05T01:00:00 -> 2023-10-05T02:00:00
  product #2: 2023-10-05T02:00:00 -> 2023-10-05T03:00:00
  product #3: 2023-10-05T03:00:00 -> 2023-10-05T04:00:00
  product #4: 2023-10-05T04:00:00 -> 2023-10-05T05:00:00
  product #5: 2023-10-05T05:00:00 -> 2023-10-05T06:00:00
  product #6: 2023-10-05T06:00:00 -> 2023-10-05T07:00:00
  product #7: 2023-10-05T07:00:00 -> 2023-10-05T08:00:00
  product #8: 2023-10-05T08:00:00 -> 2023-10-05T09:00:00
  product #9: 2023-10-05T09:00:00 -> 2023-10-05T10:00:00
  product #10: 2023-10-05T10:00:00 -> 2023-10-05T11:00:00
  product #11: 2023-10-05T11:00:00 -> 2023-10-05T12:00:00
  product #12: 2023-10-05T12:00:00 -> 2023-10-05T13:00:00
  product #13: 2023-10-05T13:00:00 -> 2023-10-05T14:00:00
  product #14: 2023-10-05T14:00:00 -> 2023-10-05T15:00:00
  product #15: 2023-10-05T15:00:00 -> 2023-10-05T16:00:00
  product #16: 2023-10-05T16:00:00 -> 2023-10-

In this section, we add a market operator to the simulation world and create a market with previously defined configuration.

In this code:
- `mo_id = "market_operator"` assigns the identifier "market_operator" to the market operator.

- `world.add_market_operator(id=mo_id)` adds a market operator to the simulation world with the specified identifier "market_operator". A market operator in this context represents an entity responsible for operating and managing one or more markets within the simulation.

- The loop `for market_config in marketdesign:` iterates over the market configurations defined in the `marketdesign` list.

  - `world.add_market(market_operator_id=mo_id, market_config=market_config)` associates each market configuration with the market operator identified by "market_operator". This effectively adds the specified market configuration to the simulation world under the management of the market operator.

## Adding Unit Operators and Units

After initializing the simulation, and creating a market, we add unit operators and units to the simulation world. A **unit** in ASSUME refers to an entity that participates in the market, either buying or selling electricity.

In [14]:
# snapshot attributes before/after
before = snapshot_world(world)
#from pprint import pprint
#print("BEFORE")
#pprint(before, width=120)

In [15]:
world.add_unit_operator("demand_operator")

demand_forecast = NaiveForecast(index, demand=-100)

world.add_unit(
    id="demand_unit",
    unit_type="demand",
    unit_operator_id="demand_operator",
    unit_params={
        "min_power": -100,
        "max_power": -1000,
        "bidding_strategies": {"EOM": "naive_eom"},
        "technology": "demand",
    },
    forecaster=demand_forecast,
)

In [16]:
# run your setup or changes, then:
after = snapshot_world(world)
print("\nAFTER")
pprint(after, width=120)

# quick diff of changed keys
changed = [k for k in after.keys() if before.get(k)!=after.get(k)]
print("\nChanged keys:", changed)
for k in changed:
    print(f"\n--- {k} ---\nBEFORE: {before[k]}\nAFTER : {after[k]}")


AFTER
{'clock': {'repr': '<mango.util.clock.ExternalClock object at 0x148dc38f0>', 'type': 'ExternalClock'},
 'container': {'repr': '<mango.container.external_coupling.ExternalSchedulingContainer object at 0x148702c00>',
               'type': 'ExternalSchedulingContainer'},
 'database_uri': {'repr': 'None', 'type': 'NoneType'},
 'db_uri': {'repr': 'sqlite:///local_db/assume_db.db', 'type': 'URL'},
 'end': {'repr': 'datetime.datetime(2023, 12, 5, 0, 0)', 'type': 'datetime'},
 'engine': {'repr': 'None', 'type': 'NoneType'},
 'index': {'repr': 'None', 'type': 'NoneType'},
 'markets': {'len': 1, 'type': 'dict'},
 'simulation_id': {'repr': "'world_script_simulation'", 'type': 'str'},
 'start': {'repr': 'datetime.datetime(2023, 10, 4, 0, 0)', 'type': 'datetime'},
 'unit_operators': {'len': 1, 'type': 'dict'}}

Changed keys: ['unit_operators']

--- unit_operators ---
BEFORE: {'type': 'dict', 'len': 0}
AFTER : {'type': 'dict', 'len': 1}


In [ ]:
print("Number of unit_operators:", len(getattr(world, "unit_operators", {})))

for uo_id, uo in getattr(world, "unit_operators", {}).items():
    print("=== unit operator key:", uo_id)
    pprint(getattr(uo, "__dict__", {}), width=120)

This code segment sets up a demand unit managed by the "demand_operator" unit operator, equipped with a naive demand forecast, and establishes its operational parameters within the electricity market simulation framework.

In this code:
- `world.add_unit_operator("demand_operator")` adds a unit operator with the identifier "demand_operator" to the simulation world. A unit operator manages a group of similar units within the simulation.

- `demand_forecast = NaiveForecast(index, demand=-100)` creates a naive demand forecast object named `demand_forecast`. This forecast is initialized with an index and a constant demand value of 100.

- `world.add_unit(...)` adds a demand unit to the simulation world with the following specifications:

  - `id="demand_unit"` assigns the identifier "demand_unit" to the demand unit.

  - `unit_type="demand"` specifies that this unit is of type "demand", indicating that it represents a consumer of electricity.

  - `unit_operator_id="demand_operator"` associates the unit with the unit operator identified as "demand_operator".

  - `unit_params` provides various parameters for the demand unit, including minimum and maximum power, bidding strategies, and technology type.

  - `forecaster=demand_forecast` associates the demand forecast (`demand_forecast`) with the demand unit, allowing the unit to utilize this forecast for its behavior within the simulation.

In [17]:
world.add_unit_operator("unit_operator")

nuclear_forecast = NaiveForecast(index, availability=1, fuel_price=3, co2_price=0.1)

world.add_unit(
    id="nuclear_unit",
    unit_type="power_plant",
    unit_operator_id="unit_operator",
    unit_params={
        "min_power": 200,
        "max_power": 1000,
        "bidding_strategies": {"EOM": "naive_eom"},
        "technology": "nuclear",
    },
    forecaster=nuclear_forecast,
)

In [18]:
from pprint import pprint

print("Number of unit_operators:", len(getattr(world, "unit_operators", {})))

for uo_id, uo in getattr(world, "unit_operators", {}).items():
    print("=== unit operator key:", uo_id)
    pprint(getattr(uo, "__dict__", {}), width=120)

Number of unit_operators: 2
=== unit operator key: demand_operator
{'_context': <mango.agent.role.RoleContext object at 0x148dea630>,
 'available_markets': [MarketConfig(market_id='EOM',
                                    opening_hours=<dateutil.rrule.rrule object at 0x148dc2990>,
                                    opening_duration=datetime.timedelta(seconds=3600),
                                    market_mechanism='pay_as_clear',
                                    market_products=[MarketProduct(duration=datetime.timedelta(seconds=3600),
                                                                   count=24,
                                                                   first_delivery=datetime.timedelta(seconds=43200),
                                                                   only_hours=None,
                                                                   eligible_lambda_function=None)],
                                    product_type='energy',
              

This code segment sets up a nuclear power plant unit managed by the "unit_operator" unit operator, equipped with a naive availability and cost forecast, and establishes its operational parameters within the electricity market simulation framework.
print("Number of unit_operators:", len(getattr(world, "unit_operators", {})))
for uo_id, uo in getattr(world, "unit_operators", {}).items():
    print("=== unit operator key:", uo_id)

    pprint(getattr(uo, "__dict__", {}), width=120)
- `nuclear_forecast = NaiveForecast(index, availability=1, fuel_price=3, co2_price=0.1)` creates a naive forecast for the nuclear power plant. This forecast is initialized with an index, a constant availability of 1, a fuel price of 3, and a CO2 price of 0.1.

- `world.add_unit(...)` adds a nuclear power plant unit to the simulation world with the following specifications:

  - `id="nuclear_unit"` assigns the identifier "nuclear_unit" to the nuclear power plant unit.

  - `unit_type="power_plant"` specifies that this unit is of type "power_plant", indicating that it represents a power generation facility.

  - `unit_operator_id="unit_operator"` associates the unit with the unit operator identified as "unit_operator".

  - `unit_params` provides various parameters for the nuclear power plant unit, including minimum and maximum power, bidding strategies, and technology type.

  - `forecaster=nuclear_forecast` associates the nuclear forecast (`nuclear_forecast`) with the nuclear power plant unit, allowing the unit to utilize this forecast for its behavior within the simulation.

## Running the Simulation

Finally, we run the simulation to observe the market behaviors and outcomes.

In [19]:
world.run()

world_script_simulation 2023-12-05 00:00:00: : 5356801.0it [00:00, 7409982.30it/s]                           


In [20]:
#print price for each auctioned interval
import sqlite3

conn = sqlite3.connect('local_db/assume_db.db')
df = pd.read_sql("SELECT * FROM market_meta WHERE simulation = ?", conn, params=(simulation_id,))
conn.close()

if not df.empty:
    print("Columns in market_meta:", df.columns.tolist())
    df = df.sort_values('time')
    first_row = df.iloc[0]
    last_row = df.iloc[-1]
    print(f"First auction (start: {first_row['time']}): price = {first_row['price']}")
    print(f"Last auction (start: {last_row['time']}): price = {last_row['price']}")
else:
    print("No market data found.")

Columns in market_meta: ['index', 'demand_volume', 'demand_volume_energy', 'market_id', 'max_price', 'min_price', 'node', 'only_hours', 'price', 'product_end', 'product_start', 'simulation', 'supply_volume', 'supply_volume_energy', 'time']
First auction (start: 2023-10-05 00:00:00.000000): price = 3
Last auction (start: 2023-12-04 23:00:00.000000): price = 3


In [21]:
import sqlite3

conn = sqlite3.connect('local_db/assume_db.db')
df = pd.read_sql("SELECT * FROM market_meta WHERE simulation = ?", conn, params=(simulation_id,))
conn.close()

df

,index,demand_volume,demand_volume_energy,market_id,max_price,min_price,node,only_hours,price,product_end,product_start,simulation,supply_volume,supply_volume_energy,time
0,0,100,100.0,EOM,3,3,None,None,3,2023-10-05 01:00:00.000000,2023-10-05 00:00:00.000000,world_script_simulation,100,100.0,2023-10-05 00:00:00.000000
1,1,100,100.0,EOM,3,3,None,None,3,2023-10-05 02:00:00.000000,2023-10-05 01:00:00.000000,world_script_simulation,100,100.0,2023-10-05 01:00:00.000000
2,2,100,100.0,EOM,3,3,None,None,3,2023-10-05 03:00:00.000000,2023-10-05 02:00:00.000000,world_script_simulation,100,100.0,2023-10-05 02:00:00.000000
3,3,100,100.0,EOM,3,3,None,None,3,2023-10-05 04:00:00.000000,2023-10-05 03:00:00.000000,world_script_simulation,100,100.0,2023-10-05 03:00:00.000000
4,4,100,100.0,EOM,3,3,None,None,3,2023-10-05 05:00:00.000000,2023-10-05 04:00:00.000000,world_script_simulation,100,100.0,2023-10-05 04:00:00.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1459,19,100,100.0,EOM,3,3,None,None,3,2023-12-04 20:00:00.000000,2023-12-04 19:00:00.000000,world_script_simulation,100,100.0,2023-12-04 19:00:00.000000
1460,20,100,100.0,EOM,3,3,None,None,3,2023-12-04 21:00:00.000000,2023-12-04 20:00:00.000000,world_script_simulation,100,100.0,2023-12-04 20:00:00.000000
1461,21,100,100.0,EOM,3,3,None,None,3,2023-12-04 22:00:00.000000,2023-12-04 21:00:00.000000,world_script_simulation,100,100.0,2023-12-04 21:00:00.000000
1462,22,100,100.0,EOM,3,3,None,None,3,2023-12-04 23:00:00.000000,2023-12-04 22:00:00.000000,world_script_simulation,100,100.0,2023-12-04 22:00:00.000000


## Conclusion

In this notebook, we have demonstrated the basic steps involved in setting up and running a simulation using the ASSUME framework for simulating electricity markets. This example is intended to provide a detailed overview of internal workings of the framework and its components. This approach can be used for small simulations with a few agents and markets. In the next notebook we will explore how this process is automated for large scale simulations using input files.

## The whole code as a single cell


In [ ]:
import logging
from datetime import datetime, timedelta

import pandas as pd
from dateutil import rrule as rr

from assume import World
from assume.common.forecasts import NaiveForecast
from assume.common.market_objects import MarketConfig, MarketProduct

log = logging.getLogger(__name__)

db_uri = "sqlite:///local_db/assume_db.db"

world = World(database_uri=db_uri)

start = datetime(2023, 1, 1)
end = datetime(2023, 3, 31)
index = pd.date_range(
    start=start,
    end=end + timedelta(hours=24),
    freq="h",
)
simulation_id = "world_script_simulation"

world.setup(
    start=start,
    end=end,
    save_frequency_hours=48,
    simulation_id=simulation_id,
    index=index,
)

marketdesign = [
    MarketConfig(
        market_id="EOM",
        opening_hours=rr.rrule(rr.HOURLY, interval=24, dtstart=start, until=end),
        opening_duration=timedelta(hours=1),
        market_mechanism="pay_as_clear",
        market_products=[MarketProduct(timedelta(hours=1), 24, timedelta(hours=1))],
        additional_fields=["block_id", "link", "exclusive_id"],
    )
]

mo_id = "market_operator"
world.add_market_operator(id=mo_id)

for market_config in marketdesign:
    world.add_market(market_operator_id=mo_id, market_config=market_config)

world.add_unit_operator("demand_operator")

demand_forecast = NaiveForecast(index, demand=-100)

world.add_unit(
    id="demand_unit",
    unit_type="demand",
    unit_operator_id="demand_operator",
    unit_params={
        "min_power": 0,
        "max_power": -1000,
        "bidding_strategies": {"EOM": "naive_eom"},
        "technology": "demand",
    },
    forecaster=demand_forecast,
)

world.add_unit_operator("unit_operator")

nuclear_forecast = NaiveForecast(index, availability=1, fuel_price=3, co2_price=0.1)

world.add_unit(
    id="nuclear_unit",
    unit_type="power_plant",
    unit_operator_id="unit_operator",
    unit_params={
        "min_power": 200,
        "max_power": 1000,
        "bidding_strategies": {"EOM": "naive_eom"},
        "technology": "nuclear",
    },
    forecaster=nuclear_forecast,
)

world.run()

In [ ]:
#print price for each auctioned interval
import sqlite3

conn = sqlite3.connect('local_db/assume_db.db')
df = pd.read_sql("SELECT * FROM market_meta WHERE simulation = ?", conn, params=(simulation_id,))
conn.close()

if not df.empty:
    print("Columns in market_meta:", df.columns.tolist())
    df = df.sort_values('time')
    first_row = df.iloc[0]
    last_row = df.iloc[-1]
    print(f"First auction (start: {first_row['time']}): price = {first_row['price']}")
    print(f"Last auction (start: {last_row['time']}): price = {last_row['price']}")
else:
    print("No market data found.")

## Conclusion

In this notebook, we have demonstrated the basic steps involved in setting up and running a simulation using the ASSUME framework for simulating electricity markets. This example is intended to provide a detailed overview of internal workings of the framework and its components. This approach can be used for small simulations with a few agents and markets. In the next notebook we will explore how this process is automated for large scale simulations using input files.